In [1]:
# =========================
# ADVANCED AUDIO CLASSIFICATION - IMPROVED FEATURES WITH DNN & CNN
# =========================
import os
import numpy as np
import librosa
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Tambahan untuk DNN dan CNN
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# ---------- Konfigurasi ----------
DATA_DIR = r"D:\web\cnn_clasification\archive\Data\genres_original"
SR = 22050
DURATION = 2.0  # Lebih pendek untuk fokus pada kata
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---------- ADVANCED FEATURE EXTRACTION ----------
def extract_advanced_features(audio_path, sr=22050, duration=2.0, for_cnn=False):
    """Feature extraction yang lebih comprehensive untuk speech recognition"""
    try:
        # Load audio dengan preprocessing
        y, sr = librosa.load(audio_path, sr=sr, duration=duration, mono=True)
        
        # Trim silence untuk fokus pada bagian yang berbunyi
        y_trimmed, _ = librosa.effects.trim(y, top_db=25)
        if len(y_trimmed) > 0:
            y = y_trimmed
        
        # Fixed length
        target_length = int(sr * duration)
        if len(y) < target_length:
            y = np.pad(y, (0, target_length - len(y)), mode='constant')
        else:
            y = y[:target_length]
        
        if for_cnn:
            # Untuk CNN, kembalikan mel spectrogram sebagai 2D array
            mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64, hop_length=512)
            mel_db = librosa.power_to_db(mel)
            # Normalize
            mel_db = (mel_db - np.min(mel_db)) / (np.max(mel_db) - np.min(mel_db))
            # Resize to fixed shape (64, 87) assuming hop_length=512 and duration=2s
            if mel_db.shape[1] < 87:
                mel_db = np.pad(mel_db, ((0, 0), (0, 87 - mel_db.shape[1])), mode='constant')
            else:
                mel_db = mel_db[:, :87]
            return mel_db
        
        features = []
        
        # 1. MFCC dengan berbagai konfigurasi (penting untuk speech)
        mfcc13 = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, hop_length=512)
        mfcc20 = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20, hop_length=512)
        
        # MFCC statistics
        features.extend(np.mean(mfcc13, axis=1))  # 13 features
        features.extend(np.std(mfcc13, axis=1))   # 13 features
        features.extend(np.mean(mfcc20, axis=1))  # 20 features
        features.extend(np.std(mfcc20, axis=1))   # 20 features
        
        # 2. Delta MFCC (perubahan MFCC over time - penting untuk speech)
        mfcc_delta = librosa.feature.delta(mfcc13)
        mfcc_delta2 = librosa.feature.delta(mfcc13, order=2)
        
        features.extend(np.mean(mfcc_delta, axis=1))   # 13 features
        features.extend(np.std(mfcc_delta, axis=1))    # 13 features
        features.extend(np.mean(mfcc_delta2, axis=1))  # 13 features
        features.extend(np.std(mfcc_delta2, axis=1))   # 13 features
        
        # 3. Spectral features
        spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
        spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
        
        features.append(np.mean(spectral_centroid))
        features.append(np.std(spectral_centroid))
        features.append(np.mean(spectral_rolloff))
        features.append(np.std(spectral_rolloff))
        features.append(np.mean(spectral_bandwidth))
        features.append(np.std(spectral_bandwidth))
        
        # 4. Chroma features (untuk karakteristik pitch)
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        features.extend(np.mean(chroma, axis=1))  # 12 features
        features.extend(np.std(chroma, axis=1))   # 12 features
        
        # 5. Spectral contrast
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        features.extend(np.mean(contrast, axis=1))  # 7 features
        features.extend(np.std(contrast, axis=1))   # 7 features
        
        # 6. Tonnetz features (harmonic content)
        tonnetz = librosa.feature.tonnetz(y=y, sr=sr)
        features.extend(np.mean(tonnetz, axis=1))  # 6 features
        features.extend(np.std(tonnetz, axis=1))   # 6 features
        
        # 7. Rhythm features
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        features.append(tempo)
        
        # 8. Zero crossing rate
        zcr = librosa.feature.zero_crossing_rate(y)
        features.append(np.mean(zcr))
        features.append(np.std(zcr))
        
        # 9. RMS energy
        rms = librosa.feature.rms(y=y)
        features.append(np.mean(rms))
        features.append(np.std(rms))
        
        # 10. Mel spectrogram statistics
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64)
        mel_db = librosa.power_to_db(mel)
        features.append(np.mean(mel_db))
        features.append(np.std(mel_db))
        features.append(np.max(mel_db))
        features.append(np.min(mel_db))
        
        # Convert to array
        features_array = np.array(features, dtype=np.float32)
        
        # Replace any NaN values
        features_array = np.nan_to_num(features_array)
        
        print(f"✅ {Path(audio_path).name}: {len(features_array)} features")
        return features_array
        
    except Exception as e:
        print(f"❌ {Path(audio_path).name}: {e}")
        return None

# ---------- ROBUST DATA COLLECTION ----------
def collect_advanced_dataset(data_dir, samples_per_class=20):
    """Kumpulkan dataset dengan advanced features"""
    data_root = Path(data_dir)
    class_dirs = [d for d in data_root.iterdir() if d.is_dir()]
    class_dirs.sort()
    class_names = [d.name for d in class_dirs]
    
    print("🎵 COLLECTING ADVANCED DATASET")
    print(f"Classes: {class_names}")
    print("=" * 50)
    
    X = []
    y = []
    X_cnn = []
    file_info = []
    
    for class_idx, class_dir in enumerate(class_dirs):
        print(f"\n📂 Processing: {class_names[class_idx]}")
        
        # Get audio files
        audio_files = []
        for ext in [".wav", ".mp3"]:
            audio_files.extend(list(class_dir.glob(f"*{ext}")))
        
        print(f"   Found {len(audio_files)} files")
        
        # Limit samples
        if len(audio_files) > samples_per_class:
            audio_files = audio_files[:samples_per_class]
        
        processed = 0
        for i, audio_file in enumerate(audio_files):
            print(f"   [{i+1}/{len(audio_files)}] {audio_file.name}")
            
            features = extract_advanced_features(str(audio_file))
            features_cnn = extract_advanced_features(str(audio_file), for_cnn=True)
            
            if features is not None and features_cnn is not None:
                X.append(features)
                X_cnn.append(features_cnn)
                y.append(class_idx)
                file_info.append(str(audio_file))
                processed += 1
        
        print(f"   🎯 Completed: {processed}/{len(audio_files)}")
    
    if len(X) == 0:
        print("❌ No features extracted! Using fallback...")
        return create_advanced_synthetic_data(class_names, samples_per_class)
    
    X_array = np.array(X)
    X_cnn_array = np.array(X_cnn)
    y_array = np.array(y)
    
    print(f"\n📊 ADVANCED DATASET CREATED!")
    print(f"   Total samples: {X_array.shape[0]}")
    print(f"   Features per sample: {X_array.shape[1]}")
    print(f"   CNN input shape: {X_cnn_array.shape[1:]}")
    print(f"   Classes: {len(class_names)}")
    
    return X_array, X_cnn_array, y_array, class_names

def create_advanced_synthetic_data(class_names, samples_per_class=20):
    """Buat synthetic data yang lebih realistic"""
    print("🔧 CREATING ADVANCED SYNTHETIC DATA")
    
    n_classes = len(class_names)
    n_features = 158  # Sesuai dengan advanced feature extraction
    total_samples = n_classes * samples_per_class
    
    # Buat data yang lebih realistic dengan class-specific patterns
    X = []
    y = []
    X_cnn = []
    
    for class_idx in range(n_classes):
        # Setiap class memiliki pattern yang sedikit berbeda
        base_mean = class_idx * 0.3
        base_std = 1.0 + class_idx * 0.1
        
        for _ in range(samples_per_class):
            features = np.random.normal(base_mean, base_std, n_features)
            # Tambahkan beberapa distinctive features
            features[class_idx * 10:(class_idx + 1) * 10] += np.random.normal(1.0, 0.2, 10)
            X.append(features)
            
            # Synthetic CNN data (random 2D array)
            cnn_features = np.random.rand(64, 87)
            X_cnn.append(cnn_features)
            y.append(class_idx)
    
    X_array = np.array(X)
    X_cnn_array = np.array(X_cnn)
    y_array = np.array(y)
    
    print(f"✅ Advanced synthetic data: {X_array.shape[0]} samples, {X_array.shape[1]} features")
    return X_array, X_cnn_array, y_array, class_names

# ---------- ADVANCED MODEL TRAINING ----------
def build_dnn_model(input_shape, num_classes):
    """Build DNN model"""
    model = keras.Sequential([
        layers.Dense(256, activation='relu', input_shape=(input_shape,)),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_cnn_model(input_shape, num_classes):
    """Build CNN model for spectrograms"""
    model = keras.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def train_advanced_models(X, X_cnn, y, class_names):
    """Train multiple advanced models including DNN and CNN"""
    print("\n🤖 TRAINING ADVANCED MODELS")
    print("=" * 50)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=SEED, stratify=y
    )
    X_cnn_train, X_cnn_test, _, _ = train_test_split(
        X_cnn, y, test_size=0.3, random_state=SEED, stratify=y
    )
    
    print(f"📈 Data Split:")
    print(f"   Training: {X_train.shape[0]} samples")
    print(f"   Testing: {X_test.shape[0]} samples")
    print(f"   Features: {X_train.shape[1]}")
    print(f"   CNN Shape: {X_cnn_train.shape[1:]}")
    
    # Scale features for ML models
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Define models
    models = {
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=SEED),
        'SVM RBF': SVC(kernel='rbf', random_state=SEED, probability=True),
        'SVM Linear': SVC(kernel='linear', random_state=SEED, probability=True),
    }
    
    # Deep Learning models
    dl_models = {
        'DNN': build_dnn_model(X_train_scaled.shape[1], len(class_names)),
        'CNN': build_cnn_model(X_cnn_train.shape[1:], len(class_names)),
    }
    
    results = {}
    best_accuracy = 0
    best_model_name = ""
    
    # Train ML models
    for name, model in models.items():
        print(f"\n🔧 Training {name}...")
        model.fit(X_train_scaled, y_train)
        
        y_pred = model.predict(X_test_scaled)
        accuracy = accuracy_score(y_test, y_pred)
        
        results[name] = {
            'model': model,
            'accuracy': accuracy,
            'predictions': y_pred,
            'history': None  # No history for ML models
        }
        
        print(f"   ✅ {name} Accuracy: {accuracy:.4f}")
        
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_model_name = name
            best_model = model
    
    # Train DL models
    for name, model in dl_models.items():
        print(f"\n🔧 Training {name}...")
        
        callbacks = [
            EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
            ModelCheckpoint(f'best_{name.lower()}_model.h5', save_best_only=True)
        ]
        
        if name == 'DNN':
            history = model.fit(
                X_train_scaled, y_train,
                epochs=100, batch_size=32,
                validation_split=0.2,
                callbacks=callbacks,
                verbose=1
            )
            y_pred = np.argmax(model.predict(X_test_scaled), axis=1)
        elif name == 'CNN':
            # Add channel dimension for CNN
            X_cnn_train_exp = np.expand_dims(X_cnn_train, axis=-1)
            X_cnn_test_exp = np.expand_dims(X_cnn_test, axis=-1)
            
            history = model.fit(
                X_cnn_train_exp, y_train,
                epochs=100, batch_size=32,
                validation_split=0.2,
                callbacks=callbacks,
                verbose=1
            )
            y_pred = np.argmax(model.predict(X_cnn_test_exp), axis=1)
        
        accuracy = accuracy_score(y_test, y_pred)
        
        results[name] = {
            'model': model,
            'accuracy': accuracy,
            'predictions': y_pred,
            'history': history
        }
        
        print(f"   ✅ {name} Accuracy: {accuracy:.4f}")
        
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_model_name = name
            best_model = model
    
    # Best model analysis
    print(f"\n🎯 BEST MODEL: {best_model_name}")
    print(f"📊 BEST ACCURACY: {best_accuracy:.4f}")
    
    best_predictions = results[best_model_name]['predictions']
    
    # Detailed classification report
    print(f"\n📈 DETAILED CLASSIFICATION REPORT:")
    print(classification_report(y_test, best_predictions, target_names=class_names, digits=4))
    
    # Confusion matrix
    plt.figure(figsize=(10, 8))
    cm = confusion_matrix(y_test, best_predictions)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - {best_model_name}\nAccuracy: {best_accuracy:.4f}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
    # Model comparison
    model_names = list(results.keys())
    accuracies = [results[name]['accuracy'] for name in model_names]
    
    plt.figure(figsize=(10, 6))
    bars = plt.bar(model_names, accuracies, color=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#0B84A5'])
    plt.title('Advanced Models Comparison')
    plt.xlabel('Models')
    plt.ylabel('Accuracy')
    plt.ylim(0, 1.0)
    plt.grid(axis='y', alpha=0.3)
    
    for bar, acc in zip(bars, accuracies):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                f'{acc:.3f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Plot training history for DL models
    for name in ['DNN', 'CNN']:
        if name in results and results[name]['history'] is not None:
            history = results[name]['history']
            plt.figure(figsize=(12, 4))
            
            plt.subplot(1, 2, 1)
            plt.plot(history.history['accuracy'], label='Train Accuracy')
            plt.plot(history.history['val_accuracy'], label='Val Accuracy')
            plt.title(f'{name} Accuracy')
            plt.xlabel('Epoch')
            plt.ylabel('Accuracy')
            plt.legend()
            
            plt.subplot(1, 2, 2)
            plt.plot(history.history['loss'], label='Train Loss')
            plt.plot(history.history['val_loss'], label='Val Loss

SyntaxError: unterminated string literal (detected at line 450) (2346954535.py, line 450)